In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 70.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


##Intitilize the model

In [2]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-4-E2B-it"

# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

##running model

In [8]:
# Prompt - add video before text
messages = [
    {
        'role': 'user',
        'content': [
            {"type": "video", "video": "/content/test squash game 2.mp4"},
            {'type': 'text', 'text': 'Describe this Squash match.'}
        ]
    }
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=512)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
processor.parse_response(response)


{'role': 'assistant',
 'content': 'This is a video of a **squash match** being played indoors on a court.\n\nHere\'s a breakdown of what can be observed from the frames:\n\n* **Sport:** The activity is clearly squash, indicated by the players holding rackets and the distinctive rectangular court.\n* **Players:** There are two players visible on the court at any given time.\n* **Action:** The players are actively engaged in the game, moving, hitting the ball, and positioning themselves on the court.\n* **Setting:** The match is taking place in a well-lit indoor facility. The court has a distinctive color scheme (grey/green surface with red boundary lines).\n* **Environment:** The background shows the walls of the venue, featuring advertisements and branding (logos like "CGCG," "ACE," and "ACEUM"), suggesting a professional or high-level tournament setting.\n* **Perspective:** The footage is captured from a high, overhead angle (likely a camera mounted high up or a drone), providing a wi

In [9]:
# Prompt - add audio before text
messages = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": "/content/Record (online-voice-recorder.com) (2).mp3"},
            {"type": "text", "text": "Transcribe the following speech segment in its original language. Follow these specific instructions for formatting the answer:\n* Only output the transcription, with no newlines.\n* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three.The one thatis talking is named Amin"},
        ]
    }
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=512)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
processor.parse_response(response)


{'role': 'assistant',
 'content': "Hello, I am Amin. Let's dive into my latest work in artificial intelligence and software engineering."}

In [6]:
# Prompt - add image before text
messages = [
    {
        "role": "user", "content": [
            {"type": "image", "url": "/content/aTeam-Romania-copy-2.jpg"},
            {"type": "text", "text": "What is shown in this image?"}
        ]
    }
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=512)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
processor.parse_response(response)


{'role': 'assistant',
 'content': 'This image appears to be taken from a **sports event**, likely a **gymnastics competition**, given the attire of the person in the center and the general setting.\n\nHere\'s a breakdown of what is visible:\n\n1.  **The Subject:** A male athlete wearing a light pink/coral t-shirt and dark shorts is in the center of the frame, appearing to be performing or preparing for a movement.\n2.  **The Venue/Background:** The background is dominated by a large, dark blue or black screen/backdrop.\n    *   It features prominent branding for **"THE WORLD GAMES 2025 CHENGDU"**.\n    *   There are bright **magenta/pink lines** framing the screen, adding a modern, high-tech aesthetic.\n    *   The overall lighting is bright and theatrical, typical of televised sporting events.\n3.  **Foreground Elements:**\n    *   The floor is a dark, perhaps wooden or specialized sports mat surface.\n    *   There are **metal railings or supports** in the foreground, suggesting the 

##Streaming

In [10]:
from transformers import TextStreamer

# Initialize the streamer
streamer = TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

# Prompt configuration (using the previous messages as an example)
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "/content/aTeam-Romania-copy-2.jpg"},
            {"type": "text", "text": "What is shown in this image? Describe it briefly."}
        ]
    }
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)

# Generate with streamer
print("Assistant: ")
_ = model.generate(
    **inputs,
    max_new_tokens=512,
    streamer=streamer
)

Assistant: 
This image appears to be taken at a large, indoor sporting event, likely a major competition, given the prominent branding.

Here is a brief description:

The image shows an indoor sports arena with a dark floor and bright, neon-like lighting, primarily in shades of blue and magenta/pink. In the background, there is a large screen displaying the logo and text for **"THE WORLD GAMES 2025 CHENGDU"**.

In the foreground, a male athlete wearing a light-colored (possibly pink or light red) t-shirt and shorts is visible, running or in motion on the court. The setting suggests a professional, high-stakes sporting event.


In [11]:
from transformers import TextStreamer

# Initialize the streamer
streamer = TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

# Prompt configuration for Audio
messages = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": "/content/Record (online-voice-recorder.com) (2).mp3"},
            {"type": "text", "text": "Transcribe the following speech segment in its original language. Follow these specific instructions for formatting the answer:\n* Only output the transcription, with no newlines.\n* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three. The one that is talking is named Amin"},
        ]
    }
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)

# Generate with streamer
print("Assistant (Streaming Transcription): ")
_ = model.generate(
    **inputs,
    max_new_tokens=512,
    streamer=streamer
)

Assistant (Streaming Transcription): 
Hello I am Amin let's dive into my latest work in artificial intelligence and software engineering.


In [12]:
from transformers import TextStreamer

# Initialize the streamer
streamer = TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

# Prompt - add video before text
messages = [
    {
        'role': 'user',
        'content': [
            {"type": "video", "video": "/content/test squash game 2.mp4"},
            {'type': 'text', 'text': 'Describe this Squash match.'}
        ]
    }
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
# Generate with streamer
print("Assistant (Streaming Transcription): ")
_ = model.generate(
    **inputs,
    max_new_tokens=512,
    streamer=streamer
)


Assistant (Streaming Transcription): 
This is a video montage or a series of shots from a **squash tennis match**, likely from a professional event given the branding visible on the court.

Here's a breakdown of what the images show:

* **Sport:** Squash.
* **Setting:** An indoor squash court with a distinctive dark grey/green playing surface and red perimeter walls/trim. The court markings (lines and circles) are clearly visible.
* **Players:** There are two players visible on the court throughout the clips.
    * One player is wearing a dark blue/purple outfit.
    * The other player is wearing a dark, possibly black or dark brown, outfit.
* **Action:** The clips capture the dynamic movements of the players during the game. We see them running, moving, and engaging in rallies. The shots are taken from a high, aerial perspective, looking down onto the court, which gives a good overview of the match progression.
* **Branding:** Various logos are visible around the perimeter of the cour